<h1 style ="text-align: center;"> Car Price Prediction: Model Building and Evaluation </h1>

## Purpose

This notebook focuses on preparing the features, training regression models, and evaluating their performance in predicting car prices. Different combinations of numerical and categorical features will be tested to determine whether the available data contains useful predictive patterns.

### Import Library 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#### Load the dataset

In [ ]:
data = pd.read_csv("Data.csv")

In [ ]:
data.head()

Further data inspection and exploratory analysis were completed in the EDA notebook. This notebook will focus on feature preparation, model training, and evaluation.

## Baseline Model Using Numerical Features

This section selects the numerical predictor variables and prepares the data for training and testing.

In [ ]:
x = data[['Year','Engine Size','Mileage']]
y = data['Price']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1 )

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(x_train, y_train)

In [ ]:
y_train_pred = baseline_model.predict(x_train)
y_test_pred = baseline_model.predict(x_test)

In [ ]:
test_mae = mean_absolute_error(y_test, y_test_pred) 
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("Baseline Linear Regression Results")
print(f"Training R²: {train_r2}")
print(f"Testing R²: {test_r2}")
print(f"Testing MAE: {test_mae}")
print(f"Testing MSE: {test_mse}")

### Baseline Model Interpretation

The baseline linear regression model produced very weak results. The training R² score was approximately 0.002, meaning that the model explained almost none of the variation in car prices within the training data. The testing R² score was slightly negative, indicating that the model performed marginally worse than simply predicting the average car price for every test observation.

The training and testing R² scores are both close to zero, so there is no clear sign of overfitting. Instead, the results suggest that the selected numerical features do not provide enough useful linear information for predicting `Price`.

The testing MAE was approximately 23,823, meaning that predictions differed from actual prices by around 23,823 price units on average. The MSE is large because it squares prediction errors, but its value is difficult to interpret directly without comparison. RMSE may therefore be used later for a more understandable error measurement.

The next experiment will test whether including categorical features improves the model's predictive performance.

## Linear Regression with Numerical and Categorical Features


This section evaluates a linear regression model using both numerical and categorical features. The categorical columns are converted into separate binary columns using one-hot encoding. The encoder is fitted only on the training data and then applied to the testing data to reduce the risk of data leakage.

In [ ]:
cat_col = ['Brand', 'Fuel Type', 'Transmission', 'Condition', 'Model']
num_col = ['Year','Engine Size','Mileage']


x = data[num_col + cat_col]
y = data['Price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('numerical', 'passthrough', num_col),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_col)
    ]
)


In [ ]:
x_train_encod = preprocessor.fit_transform(x_train)
x_test_encod = preprocessor.transform(x_test)

In [ ]:
encoded_model = LinearRegression()

In [ ]:
encoded_model.fit(x_train_encod, y_train)

In [ ]:
y_train_encoded_pred = encoded_model.predict(x_train_encod)
y_test_encoded_pred = encoded_model.predict(x_test_encod)

In [ ]:
test_mae = mean_absolute_error(y_test, y_test_encoded_pred) 
test_mse = mean_squared_error(y_test, y_test_encoded_pred)
test_r2 = r2_score(y_test, y_test_encoded_pred)
train_r2 = r2_score(y_train, y_train_encoded_pred)

print("Combined Feature Linear Regression Results")
print(f"Training R²: {train_r2}")
print(f"Testing R²: {test_r2}")
print(f"Testing MAE: {test_mae}")
print(f"Testing MSE: {test_mse}")

### Combined Feature Model Interpretation

The linear regression model using both numerical and one-hot encoded categorical features also produced very weak results. The training R² remained close to zero, while the testing R² was slightly negative. This shows that the model was unable to explain the variation in car prices in either the training or testing data.

There is no clear evidence of overfitting because both training and testing performance are similarly poor. The testing MAE and MSE showed a small improvement compared with the previous experiment, but the improvement was not large enough to make the model useful.

Overall, combining the numerical and categorical features did not produce meaningful predictive performance. These results support the earlier EDA findings that the available features contain very little useful relationship with the target variable, `Price`.

## Polynomial Regression Using Numerical Features


The linear regression experiments did not produce meaningful predictive results. Therefore, this section tests whether nonlinear relationships between the numerical features and `Price` can improve model performance.

Polynomial regression is applied using only the numerical features. If this experiment shows a clear improvement, categorical features may later be included for further evaluation.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
polynomial_features = PolynomialFeatures(degree=3)
polynomial_model = LinearRegression()

In [ ]:
poly_x = data[['Year', 'Engine Size', 'Mileage']]
poly_y = data['Price']

poly_x_train, poly_x_test, poly_y_train, poly_y_test = train_test_split(poly_x, poly_y, test_size=0.3,random_state=1)

In [ ]:
poly_x_train_transformed = polynomial_features.fit_transform(poly_x_train)
poly_x_test_transformed = polynomial_features.transform(poly_x_test)

In [ ]:
poly_scaler = StandardScaler()

poly_x_train_scaled = poly_scaler.fit_transform(poly_x_train_transformed)

poly_x_test_scaled = poly_scaler.transform(poly_x_test_transformed)

In [ ]:
polynomial_model = LinearRegression()

polynomial_model.fit( poly_x_train_scaled, poly_y_train)

In [ ]:
poly_y_train_pred = polynomial_model.predict(poly_x_train_scaled)

poly_y_test_pred = polynomial_model.predict(poly_x_test_scaled)

In [ ]:
poly_train_r2 = r2_score(poly_y_train, poly_y_train_pred)
poly_test_r2 = r2_score(poly_y_test, poly_y_test_pred)

poly_test_mae = mean_absolute_error(poly_y_test, poly_y_test_pred)
poly_test_mse = mean_squared_error(poly_y_test, poly_y_test_pred)

print("Polynomial Regression Results")
print(f"Training R²: {poly_train_r2}")
print(f"Testing R²: {poly_test_r2}")
print(f"Testing MAE: {poly_test_mae}")
print(f"Testing MSE: {poly_test_mse}")

### Polynomial Regression Interpretation

The degree-three polynomial regression model produced only a very small improvement in training performance, with a training R² of approximately 0.011. However, the testing R² remained slightly negative at approximately -0.007, meaning that the model still performed worse than simply predicting the average car price for the test data.

The testing MAE and MSE also remained very close to the earlier linear regression results. Therefore, introducing nonlinear polynomial relationships between the numerical features did not meaningfully improve prediction performance.

Across the numerical baseline, categorical model, combined-feature model, and polynomial regression experiment, the results remained consistently weak. This strongly suggests that the available dataset contains little useful predictive relationship between the provided features and `Price`.

## Final Conclusion

The exploratory data analysis showed no meaningful relationship between the available numerical or categorical features and the target variable, `Price`. This finding was further supported by the modelling experiments.

The numerical baseline model, combined-feature linear regression model, and polynomial regression model all produced R² scores close to zero or slightly negative. Their prediction errors also remained consistently high, with no meaningful improvement across the experiments.

These results indicate that the available features contain very little useful predictive information about car prices. Therefore, the dataset is not suitable for building a reliable car price prediction model in its current form, and further model tuning is unlikely to provide meaningful improvement.

For this reason, no additional modelling will be performed. A more suitable dataset containing realistic and meaningful relationships between vehicle characteristics and price would be required to continue this project successfully.